# 02. v10 피처 계산 (엣지 17 · 노드 19)

01 의 `transactions.parquet` / `kyc.parquet` 로 money_laundry 추론서버 `models/binary_v10/preprocess.py` 와
**같은 SQL**을 단계별로 실행한다. 배치 버전은 `02_build_features.py`, SQL 정의는 `v10_features.py`.
HI-Small 2022-09-01~03 구간에서 원본 `prepare_snapshot` 과 36개 피처 전부 일치(max|diff|=0) 확인함.

실험용으로 뺀 인자:
- `EXCLUDE` 시간 창의 끝. 원본 `'1 minute'` → 같은 분 거래를 '이전'으로 안 셈
- `HOUR_WINDOW` `*_count_1h` 창 길이 (원본 `'60 minutes'`)
- `CONTEXT_END` 이 날짜까지만 (원본 `ts::DATE <= context_end`)
- `WINDOW_DAYS` `context_end - N일` 이후만 (백엔드 `materials.py` 에 하한을 둘 때의 효과)
- `STATS` `"package"`(money_laundry 보정표) / `"from-data"`(이 데이터로 근사)

## 0. 설정

In [ ]:
from pathlib import Path
import json, time
import duckdb, pandas as pd, numpy as np

DATASET = "HI-Small"
TAG = None                      # 출력 폴더 접미사: features_<TAG>
EXCLUDE = "1 minute"            # 실험: "0 second" (같은 분 포함 — 단, 자기 자신도 포함됨)
HOUR_WINDOW = "60 minutes"
CONTEXT_END = None              # 예: "2022-09-10"
WINDOW_DAYS = None              # 예: 7
STATS = "package"               # "package" | "from-data"
THREADS, MEMORY, WORK_DIR = 4, "6GB", None

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "datasets").is_dir())   # 저장소 루트 (datasets/ 가 있는 곳)
BASE = ROOT / "parquets" / DATASET         # 01 출력: money_laundry_ai_/parquets/<DATASET>/
OUT = BASE / ("features" if not TAG else f"features_{TAG}"); OUT.mkdir(parents=True, exist_ok=True)
ASSETS = ROOT / "assets" / "binary_v10"      # v10 보정표 사본 (currency_stats.parquet, amount_stats.parquet)
assert (BASE / "transactions.parquet").is_file(), "01 먼저 실행"
q = lambda v: "'" + str(v).replace("'", "''") + "'"

work = (Path(WORK_DIR) if WORK_DIR else OUT) / "work_features.duckdb"
for stale in (work, Path(str(work) + ".wal")): stale.unlink(missing_ok=True)
con = duckdb.connect(str(work))
con.execute(f"SET threads={THREADS}"); con.execute(f"SET memory_limit='{MEMORY}'"); con.execute("SET preserve_insertion_order=false")

NODE_FEATURES = ["log_out_count","log_in_count","log_out_neighbors","log_in_neighbors","log_self_count",
    "log_reciprocal_neighbors","mean_paid_currency_z","paid_outlier_fraction","log_currency_count","log_active_days",
    "type_Corporation","type_Partnership","type_Sole Proprietorship","type_Individual","type_Country","type_Direct",
    "log_entity_observed_accounts","log_entity_out_diversity","log_entity_in_diversity"]
EDGE_FEATURES = ["pair_prior_count","pair_gap_minutes","pair_count_1h","sender_gap_minutes","sender_count_1h",
    "receiver_gap_minutes","receiver_count_1h","paid_currency_z","pair_history_count","reverse_history_count",
    "pair_age_at_day_start","same_entity","same_known_country","both_countries_known","same_currency",
    "is_self_transfer","amt_z_pay_ccy"]
ENTITY_TYPES = ["Corporation","Partnership","Sole Proprietorship","Individual","Country","Direct"]
meta = dict(dataset=DATASET, exclude=EXCLUDE, hour_window=HOUR_WINDOW, window_days=WINDOW_DAYS, stats=STATS)

## 1. 입력 `tx`, `kyc`

원본 `prepare_snapshot` 과 같이 중복 대표만, `day <= context_end`. `kyc` 의 NULL 은 `'UNKNOWN'` 으로 (원본 `coalesce(entity_type,'UNKNOWN')`).

In [ ]:
end = CONTEXT_END or str(con.execute(f"SELECT max(day) FROM read_parquet({q(BASE/'transactions.parquet')})").fetchone()[0])
lower = f"AND day >= DATE {q(end)} - INTERVAL {WINDOW_DAYS} DAY" if WINDOW_DAYS else ""
con.execute(f"""
    CREATE OR REPLACE TABLE tx AS
    SELECT edge_id, ts, src, dst, paid, received, payment_currency, receiving_currency, payment_format, duplicate_group_id, day, is_laundering
    FROM read_parquet({q(BASE/'transactions.parquet')})
    WHERE (duplicate_group_id IS NULL OR edge_id = duplicate_group_id) AND day <= DATE {q(end)} {lower}
    ORDER BY edge_id""")
con.execute(f"""
    CREATE OR REPLACE TABLE kyc AS
    SELECT node_id, entity_id, coalesce(entity_type, 'UNKNOWN') AS entity_type, coalesce(country, 'UNKNOWN') AS country, bank_id, account
    FROM read_parquet({q(BASE/'kyc.parquet')})""")
meta["context_end"] = end
meta["rows"], meta["day_min"], meta["day_max"] = con.execute("SELECT count(*), min(day), max(day) FROM tx").fetchone()
print(f"거래 {meta['rows']:,}  기간 {meta['day_min']} ~ {meta['day_max']}")

## 2. 보정표 `stats` / `amount_stats`

학습 때 고정한 통화별 금액 분포. 추론 때 다시 계산하지 않는다.
- `stats(currency, scale, med, iqr)` → `paid_currency_z = clip((ln(1+paid/scale) − med)/iqr, ±8)` (강건 z)
- `amount_stats(payment_currency, m, sd)` → `amt_z_pay_ccy = clip((ln(1+paid) − m)/(sd+1e-6), ±10)` (평균/표준편차 z)

`STATS="from-data"` 는 팀 패키지가 어떻게 만들었는지 모르는 상태의 **근사**(중앙값·IQR·평균·표준편차)다.

In [ ]:
if STATS == "package":
    con.execute(f"CREATE OR REPLACE TABLE stats AS SELECT * FROM read_parquet({q(ASSETS/'currency_stats.parquet')})")
    con.execute(f"CREATE OR REPLACE TABLE amount_stats AS SELECT * FROM read_parquet({q(ASSETS/'amount_stats.parquet')})")
else:
    con.execute("""
        CREATE OR REPLACE TABLE stats AS
        WITH s AS (SELECT payment_currency AS currency, quantile_cont(paid, 0.5) AS scale FROM tx GROUP BY 1),
             z AS (SELECT t.payment_currency AS currency, ln(1 + t.paid / s.scale) AS v FROM tx t JOIN s ON s.currency = t.payment_currency)
        SELECT z.currency, any_value(s.scale) AS scale, quantile_cont(v, 0.5) AS med,
               greatest(quantile_cont(v, 0.75) - quantile_cont(v, 0.25), 1e-6) AS iqr
        FROM z JOIN s USING (currency) GROUP BY z.currency""")
    con.execute("CREATE OR REPLACE TABLE amount_stats AS SELECT payment_currency, avg(ln(1+paid)) AS m, stddev_samp(ln(1+paid)) AS sd FROM tx GROUP BY 1")
con.execute(f"COPY stats TO {q(OUT/'stats.parquet')} (FORMAT PARQUET)"); con.execute(f"COPY amount_stats TO {q(OUT/'amount_stats.parquet')} (FORMAT PARQUET)")
display(con.execute("SELECT * FROM stats ORDER BY currency").df())
con.execute("SELECT * FROM amount_stats ORDER BY payment_currency").df()

In [ ]:
def amount_z(column, prefix):
    """원본 amount(): 통화 보정 강건 z-점수, ±8 클리핑. 보정표에 없는 통화 → 0"""
    return (f"CASE WHEN {prefix}.scale IS NULL THEN 0 "
            f"ELSE greatest(-8, least(8, (ln(1 + t.{column}::DOUBLE / {prefix}.scale) - {prefix}.med) / {prefix}.iqr)) END")

## 3. 엣지 피처 ① — 날짜 단위 이력 `pair_state` → `pairs`

`(src, dst, day)` 로 묶어 그 날까지 누적 건수·마지막 시각을 만들고, `ASOF JOIN … t.day > f.day` 로 **어제까지**의 값을 붙인다.
- `pair_n` 같은 쌍 어제까지 누적 건수 → `pair_history_count = ln(1+pair_n)`
- `reverse_n` 역방향(dst→src) → `reverse_history_count`
- `age` 마지막 과거 거래 → 오늘 00:00 까지 분 (없으면 −1) → `pair_age_at_day_start`

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE pair_state AS
    SELECT src, dst, day,
           sum(n)       OVER (PARTITION BY src, dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS history_count,
           max(last_ts) OVER (PARTITION BY src, dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS last_ts
    FROM (SELECT src, dst, day, count(*) n, max(ts) last_ts FROM tx GROUP BY src, dst, day)""")
con.execute("""
    CREATE OR REPLACE TABLE pairs AS
    SELECT t.src, t.dst, t.day, coalesce(f.history_count, 0) pair_n, coalesce(r.history_count, 0) reverse_n,
           CASE WHEN f.last_ts IS NULL THEN -1 ELSE date_diff('minute', f.last_ts, t.day::TIMESTAMP) END age
    FROM (SELECT DISTINCT src, dst, day FROM tx) t
    ASOF LEFT JOIN pair_state f ON t.src = f.src AND t.dst = f.dst AND t.day > f.day
    ASOF LEFT JOIN pair_state r ON t.src = r.dst AND t.dst = r.src AND t.day > r.day""")
con.execute("SELECT * FROM pairs WHERE pair_n > 0 ORDER BY pair_n DESC LIMIT 5").df()

## 4. 엣지 피처 ② — 분 단위 시간 창 `temporal`

세 파티션(`pair`=src,dst / `sender`=src / `receiver`=dst) × 세 값:
- `*_prior` : `UNBOUNDED PRECEDING ~ EXCLUDE PRECEDING` 건수 (과거 전체, 단 같은 분 제외)
- `*_last`  : 그 창의 `max(ts)` → gap 계산용
- `*_hour`  : `HOUR_WINDOW PRECEDING ~ EXCLUDE PRECEDING` 건수

`RANGE` 창이라 행 수가 아닌 `ts` 값 기준이고 양끝 포함. 원본 `EXCLUDE='1 minute'` 이면 같은 분 거래는 어느 쪽에도 안 들어간다
(분 단위 데이터라 같은 분 안의 순서를 모르기 때문).

In [ ]:
windows = []
for name, part in [("pair", "src, dst"), ("sender", "src"), ("receiver", "dst")]:
    windows += [
        f"count(*) OVER (PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '{EXCLUDE}' PRECEDING) AS {name}_prior",
        f"max(ts)  OVER (PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '{EXCLUDE}' PRECEDING) AS {name}_last",
        f"count(*) OVER (PARTITION BY {part} ORDER BY ts RANGE BETWEEN INTERVAL '{HOUR_WINDOW}' PRECEDING AND INTERVAL '{EXCLUDE}' PRECEDING) AS {name}_hour",
    ]
t0 = time.perf_counter()
con.execute("CREATE OR REPLACE TABLE temporal AS SELECT edge_id, " + ", ".join(windows) + " FROM tx")
print(f"{time.perf_counter()-t0:.1f}s")
# 한 송신 계좌의 타임라인으로 창 동작 확인
node = con.execute("SELECT src FROM tx GROUP BY src HAVING count(*) BETWEEN 6 AND 12 ORDER BY src LIMIT 1").fetchone()[0]
con.execute(f"""SELECT t.edge_id, t.ts, t.dst, z.sender_prior, z.sender_last, z.sender_hour
                FROM tx t JOIN temporal z USING (edge_id) WHERE t.src = {node} ORDER BY t.ts""").df()

## 5. 엣지 피처 ③ — 17개 조립

| # | 피처 | 식 |
|---|---|---|
| 1 | pair_prior_count | ln(1+pair_prior) |
| 2 | pair_gap_minutes | pair_last 없으면 0, 있으면 ln(2+분차) |
| 3 | pair_count_1h | ln(1+pair_hour) |
| 4–5 | sender_gap_minutes / sender_count_1h | src 기준 |
| 6–7 | receiver_gap_minutes / receiver_count_1h | dst 기준 |
| 8 | paid_currency_z | amount_z (stats) |
| 9–10 | pair_history_count / reverse_history_count | ln(1+pair_n), ln(1+reverse_n) |
| 11 | pair_age_at_day_start | age<0→0, else ln(2+age) |
| 12 | same_entity | 두 계좌 소유주 같음 |
| 13–14 | same_known_country / both_countries_known | UNKNOWN 아닐 때만 |
| 15 | same_currency | payment = receiving |
| 16 | is_self_transfer | src = dst |
| 17 | amt_z_pay_ccy | (ln(1+paid)−m)/(sd+1e-6), ±10 |

In [ ]:
gap = lambda n: f"CASE WHEN z.{n}_last IS NULL THEN 0 ELSE ln(2 + date_diff('minute', z.{n}_last, t.ts)) END"
expressions = [
    "ln(1 + z.pair_prior)", gap("pair"), "ln(1 + z.pair_hour)", gap("sender"), "ln(1 + z.sender_hour)",
    gap("receiver"), "ln(1 + z.receiver_hour)", amount_z("paid", "ps"), "ln(1 + p.pair_n)", "ln(1 + p.reverse_n)",
    "CASE WHEN p.age < 0 THEN 0 ELSE ln(2 + p.age) END",
    "CASE WHEN sk.entity_id IS NOT NULL AND sk.entity_id = dk.entity_id THEN 1 ELSE 0 END",
    "CASE WHEN sk.country != 'UNKNOWN' AND dk.country != 'UNKNOWN' AND sk.country = dk.country THEN 1 ELSE 0 END",
    "CASE WHEN sk.country != 'UNKNOWN' AND dk.country != 'UNKNOWN' THEN 1 ELSE 0 END",
    "CASE WHEN t.payment_currency = t.receiving_currency THEN 1 ELSE 0 END",
    "CASE WHEN t.src = t.dst THEN 1 ELSE 0 END",
    "CASE WHEN a.m IS NULL OR a.sd IS NULL THEN 0 ELSE greatest(-10, least(10, (ln(1 + t.paid::DOUBLE) - a.m) / (a.sd + 0.000001))) END",
]
features = ", ".join(f'({v})::FLOAT AS "{n}"' for n, v in zip(EDGE_FEATURES, expressions))
raw = "z.pair_prior, z.pair_last, z.pair_hour, z.sender_prior, z.sender_last, z.sender_hour, z.receiver_prior, z.receiver_last, z.receiver_hour, p.pair_n, p.reverse_n, p.age"
t0 = time.perf_counter()
con.execute(f"""
    CREATE OR REPLACE TABLE edge_features AS
    SELECT t.edge_id, t.ts, t.src, t.dst, {features}, {raw}, t.is_laundering, t.paid, t.payment_currency, t.payment_format
    FROM tx t JOIN temporal z USING (edge_id)
    JOIN pairs p ON t.src = p.src AND t.dst = p.dst AND t.day = p.day
    JOIN kyc sk ON sk.node_id = t.src JOIN kyc dk ON dk.node_id = t.dst
    LEFT JOIN stats ps ON ps.currency = t.payment_currency
    LEFT JOIN amount_stats a ON a.payment_currency = t.payment_currency
    ORDER BY t.edge_id""")
con.execute(f"COPY (SELECT * FROM edge_features ORDER BY edge_id) TO {q(OUT/'edge_features.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f"{time.perf_counter()-t0:.1f}s")
bad = " OR ".join(f'"{n}" IS NULL OR NOT isfinite("{n}")' for n in EDGE_FEATURES)
meta["edge_rows"] = con.execute("SELECT count(*) FROM edge_features").fetchone()[0]
meta["edge_nonfinite"] = con.execute(f"SELECT count(*) FROM edge_features WHERE {bad}").fetchone()[0]
print(f"{meta['edge_rows']:,} 행, 비유한값 {meta['edge_nonfinite']}")
con.execute("SELECT edge_id, " + ", ".join(f'"{n}"' for n in EDGE_FEATURES) + " FROM edge_features LIMIT 5").df().T

### 라벨별 분포 한눈에 — 피처가 0인 비율, 평균

In [ ]:
zero = con.execute("SELECT is_laundering, " + ", ".join(f'avg(("{n}" = 0)::INT) AS "{n}"' for n in EDGE_FEATURES) +
                   " FROM edge_features GROUP BY 1 ORDER BY 1").df().set_index("is_laundering").T
mean = con.execute("SELECT is_laundering, " + ", ".join(f'avg("{n}") AS "{n}"' for n in EDGE_FEATURES) +
                   " FROM edge_features GROUP BY 1 ORDER BY 1").df().set_index("is_laundering").T
meta["edge_zero_fraction"] = zero.mean(axis=1).round(4).to_dict()
pd.concat({"zero_rate": zero, "mean": mean}, axis=1).round(3)

## 6. 노드 피처 — `hist` 뷰 → 집계 → 19개

**집계 범위는 `tx` 전체**(context_end 까지). 대상 거래 시점과 무관한 "계좌 전체 프로필"이다 (원본 build_nodes 와 같음).
`hist` 는 tx 에 통화 z-점수와 양쪽 소유주 ID 를 붙인 뷰.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW hist AS
    SELECT t.*, {amount_z('paid','ps')} AS paid_currency_z, {amount_z('received','rs')} AS received_currency_z,
           ks.entity_id AS source_entity, kd.entity_id AS target_entity
    FROM tx t JOIN kyc ks ON ks.node_id = t.src JOIN kyc kd ON kd.node_id = t.dst
    LEFT JOIN stats ps ON ps.currency = t.payment_currency LEFT JOIN stats rs ON rs.currency = t.receiving_currency""")
t0 = time.perf_counter()
con.execute("""
    CREATE OR REPLACE TABLE node_agg AS
    WITH outgoing AS (SELECT src AS node_id, count(*) AS out_n, count(DISTINCT dst) AS out_u, count(*) FILTER (WHERE src = dst) AS self_n,
                             avg(paid_currency_z) AS out_z, avg(CASE WHEN abs(paid_currency_z) >= 3 THEN 1.0 ELSE 0 END) AS out_tail FROM hist GROUP BY src),
         incoming AS (SELECT dst AS node_id, count(*) AS in_n, count(DISTINCT src) AS in_u, avg(received_currency_z) AS in_z,
                             avg(CASE WHEN abs(received_currency_z) >= 3 THEN 1.0 ELSE 0 END) AS in_tail FROM hist GROUP BY dst),
         pairs AS (SELECT DISTINCT src, dst FROM hist WHERE src != dst),
         reciprocal AS (SELECT a.src AS node_id, count(*) AS reciprocal_n FROM pairs a JOIN pairs b ON a.src = b.dst AND a.dst = b.src GROUP BY a.src),
         currency_set AS (SELECT src AS node_id, payment_currency AS currency FROM hist UNION SELECT dst, receiving_currency FROM hist),
         currencies AS (SELECT node_id, count(*) AS n_currency FROM currency_set GROUP BY node_id),
         active AS (SELECT src AS node_id, day FROM hist UNION SELECT dst, day FROM hist),
         active_days AS (SELECT node_id, count(*) AS n_days FROM active GROUP BY node_id),
         ent_pairs AS (SELECT DISTINCT source_entity, target_entity FROM hist WHERE source_entity IS NOT NULL AND target_entity IS NOT NULL),
         ent_out AS (SELECT source_entity AS entity_id, count(*) AS ent_out_n FROM ent_pairs GROUP BY 1),
         ent_in  AS (SELECT target_entity AS entity_id, count(*) AS ent_in_n  FROM ent_pairs GROUP BY 1)
    SELECT k.*, CASE WHEN k.entity_id IS NULL THEN 0 ELSE count(*) OVER (PARTITION BY k.entity_id) END AS observed_entity_accounts,
           coalesce(o.out_n,0) out_n, coalesce(i.in_n,0) in_n, coalesce(o.out_u,0) out_u, coalesce(i.in_u,0) in_u,
           coalesce(o.self_n,0) self_n, coalesce(r.reciprocal_n,0) reciprocal_n, coalesce(o.out_z,0) out_z, coalesce(i.in_z,0) in_z,
           coalesce(o.out_tail,0) out_tail, coalesce(i.in_tail,0) in_tail, cu.n_currency, ad.n_days,
           coalesce(eo.ent_out_n,0) ent_out_n, coalesce(ei.ent_in_n,0) ent_in_n
    FROM active_days ad JOIN kyc k USING (node_id) LEFT JOIN outgoing o USING (node_id) LEFT JOIN incoming i USING (node_id)
    LEFT JOIN reciprocal r USING (node_id) JOIN currencies cu USING (node_id)
    LEFT JOIN ent_out eo ON k.entity_id = eo.entity_id LEFT JOIN ent_in ei ON k.entity_id = ei.entity_id""")
print(f"{time.perf_counter()-t0:.1f}s")
con.execute("SELECT * FROM node_agg ORDER BY out_u DESC LIMIT 5").df()

원본은 22개를 만들고 `feature_schema.node_features` 순서로 19개만 뽑는다 (`net_count_ratio`, `reciprocity_ratio`, `mean_received_currency_z`, `received_outlier_fraction` 중 3개가 빠짐 — 여기선 22개 전부 남겨 둔다).
`type_*` 6개는 one-hot, `UNKNOWN` 이면 전부 0.

In [ ]:
one_hot = ", ".join(f'(entity_type = {q(t)})::FLOAT AS "type_{t}"' for t in ENTITY_TYPES)
con.execute(f"""
    CREATE OR REPLACE TABLE node_features AS
    SELECT node_id, entity_id, entity_type, country,
           ln(1+out_n)::FLOAT log_out_count, ln(1+in_n)::FLOAT log_in_count, ln(1+out_u)::FLOAT log_out_neighbors, ln(1+in_u)::FLOAT log_in_neighbors,
           ln(1+self_n)::FLOAT log_self_count, ln(1+reciprocal_n)::FLOAT log_reciprocal_neighbors,
           ((out_n-in_n)/greatest(out_n+in_n,1))::FLOAT net_count_ratio, (reciprocal_n/greatest(out_u+in_u,1))::FLOAT reciprocity_ratio,
           out_z::FLOAT mean_paid_currency_z, in_z::FLOAT mean_received_currency_z, out_tail::FLOAT paid_outlier_fraction, in_tail::FLOAT received_outlier_fraction,
           ln(1+n_currency)::FLOAT log_currency_count, ln(1+n_days)::FLOAT log_active_days,
           {one_hot},
           ln(1+observed_entity_accounts)::FLOAT log_entity_observed_accounts, ln(1+ent_out_n)::FLOAT log_entity_out_diversity, ln(1+ent_in_n)::FLOAT log_entity_in_diversity,
           out_n, in_n, out_u, in_u, self_n, reciprocal_n, n_currency, n_days, observed_entity_accounts, ent_out_n, ent_in_n
    FROM node_agg ORDER BY node_id""")
con.execute(f"COPY (SELECT * FROM node_features ORDER BY node_id) TO {q(OUT/'node_features.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")
bad = " OR ".join(f'"{n}" IS NULL OR NOT isfinite("{n}")' for n in NODE_FEATURES)
meta["node_rows"] = con.execute("SELECT count(*) FROM node_features").fetchone()[0]
meta["node_nonfinite"] = con.execute(f"SELECT count(*) FROM node_features WHERE {bad}").fetchone()[0]
print(f"{meta['node_rows']:,} 노드, 비유한값 {meta['node_nonfinite']}")
con.execute("SELECT node_id, entity_type, " + ", ".join(f'"{n}"' for n in NODE_FEATURES) + " FROM node_features ORDER BY log_out_neighbors DESC LIMIT 5").df().T

### 세탁 거래에 연루된 계좌 vs 그 외 — 노드 피처 평균

In [ ]:
con.execute("""CREATE OR REPLACE VIEW laundering_nodes AS
               SELECT DISTINCT node_id FROM (SELECT src AS node_id FROM tx WHERE is_laundering UNION ALL SELECT dst FROM tx WHERE is_laundering)""")
con.execute("SELECT (n.node_id IN (SELECT node_id FROM laundering_nodes)) AS laundering_involved, count(*) nodes, " +
            ", ".join(f'avg("{f}") AS "{f}"' for f in NODE_FEATURES) +
            " FROM node_features n GROUP BY 1 ORDER BY 1").df().set_index("laundering_involved").T.round(3)

## 7. 마무리 — `features.json`, 작업 파일 정리

In [ ]:
con.close()
for stale in (work, Path(str(work) + ".wal")): stale.unlink(missing_ok=True)
(OUT / "features.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
for f in sorted(OUT.glob("*")):
    print(f"{f.name:24s} {f.stat().st_size/1e6:9.1f} MB")

## 다음: 03 실험

같은 폴더에 노트북/스크립트를 만들고 `data/<dataset>/features*/edge_features.parquet`, `node_features.parquet`, `patterns.parquet` 를 읽으면 된다.
창을 바꿔 다시 계산할 때는 위 `0. 설정` 의 `EXCLUDE`, `WINDOW_DAYS`, `TAG` 만 바꾸고 전체 실행하거나,
`v10_features.build_edge_features(con, exclude=..., hour_window=...)` 를 import 해서 쓴다.